In [ ]:
import importlib, os

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except Exception:
        !pip -q install {pkg}

ensure("rasterio")
ensure("geopandas")
ensure("pyproj")
ensure("shapely")
ensure("tqdm")

import pandas as pd
import numpy as np
import glob, re

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive"

DHS_ZIP = os.path.join(DRIVE_DIR, "IN_2015_DHS.zip")

# VIIRS tile (2015, 75N060E)
NL_TGZ = os.path.join(
    DRIVE_DIR,
    "SVDNB_npp_20150101-20151231_75N060E_v10_c201701311200.tgz"
)

## Unzip DHS and copy the nightlight data to the Colab


In [ ]:
import zipfile, shutil, glob

BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data")
DHS_DIR  = os.path.join(DATA_DIR, "dhs", "IN_2015_DHS")
NL_DIR   = os.path.join(DATA_DIR, "nightlights")
RES_DIR  = os.path.join(BASE_DIR, "results")

os.makedirs(os.path.join(DATA_DIR, "dhs"), exist_ok=True)
os.makedirs(NL_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

# Unzip DHS survey data
if not os.path.exists(DHS_DIR) or (os.path.isdir(DHS_DIR) and len(os.listdir(DHS_DIR)) == 0):
    with zipfile.ZipFile(DHS_ZIP, "r") as z:
        z.extractall(os.path.join(DATA_DIR, "dhs"))
    print("DHS data extracted to:", DHS_DIR)
else:
    print("DHS directory already exists, skipping extraction.")

# Copy VIIRS nightlights file
local_tgz = os.path.join(NL_DIR, os.path.basename(NL_TGZ))
if not os.path.exists(local_tgz):
    shutil.copy(NL_TGZ, local_tgz)
    print("Nightlights file copied to:", local_tgz)
else:
    print("Nightlights file already present, skipping copy.")

## Unzip the tgz file and locate the GeoTIFF

In [ ]:
import tarfile
from pyproj import Transformer
import rasterio

# Extract tile identifier from filename
tgz_name = os.path.basename(local_tgz)
m = re.search(r"(\d{2}N\d{3}E)", tgz_name)
tile_tag = m.group(1) if m else "unknown"

NL_EXTRACT_DIR = os.path.join(NL_DIR, f"extracted_2015_{tile_tag}")
os.makedirs(NL_EXTRACT_DIR, exist_ok=True)

# Extract the archive only if no .tif files are present yet
tif_candidates = glob.glob(os.path.join(NL_EXTRACT_DIR, "**", "*.tif"), recursive=True)

if len(tif_candidates) == 0:
    with tarfile.open(local_tgz, "r:gz") as tar:
        tar.extractall(NL_EXTRACT_DIR)

    tif_candidates = glob.glob(os.path.join(NL_EXTRACT_DIR, "**", "*.tif"), recursive=True)

# Prefer the average radiance nightlights raster
preferred = []
for p in tif_candidates:
    name = os.path.basename(p).lower()
    if ("vcm-orm-ntl" in name) and name.endswith("avg_rade9.tif"):
        preferred.append(p)

if len(preferred) == 0:
    raise FileNotFoundError(
        f"Could not find 'vcm-orm-ntl' avg_rade9.tif in {NL_EXTRACT_DIR}. "
        "Please check the VIIRS tgz contents."
    )

tif_path = preferred[0]

## Automatic positioning of GPS and Household files

In [ ]:
gps_candidates = glob.glob(os.path.join(DHS_DIR, "**", "*.shp"), recursive=True)

hh_candidates = (
    glob.glob(os.path.join(DHS_DIR, "**", "*HR*.DTA"), recursive=True) +
    glob.glob(os.path.join(DHS_DIR, "**", "*HR*.dta"), recursive=True)
)

# Select the main GPS and household recode files
GPS_FILE = next(
    (p for p in gps_candidates if os.path.basename(p).upper().startswith("IAGE")),
    gps_candidates[0] if gps_candidates else None
)

HH_FILE = next(
    (p for p in hh_candidates if os.path.basename(p).upper().startswith("IAHR")),
    hh_candidates[0] if hh_candidates else None
)

In [ ]:
print(gdf.columns)

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(GPS_FILE)

# Build cluster_df
cluster_df = pd.DataFrame({
    "cluster_id": pd.to_numeric(gdf["DHSCLUST"], errors="coerce"),
    "cluster_lat": pd.to_numeric(gdf["LATNUM"], errors="coerce"),
    "cluster_lon": pd.to_numeric(gdf["LONGNUM"], errors="coerce"),
})

cluster_df = cluster_df.dropna().drop_duplicates("cluster_id")
cluster_df["cluster_id"] = cluster_df["cluster_id"].astype(int)

print("cluster_df rows:", len(cluster_df))
cluster_df.head()

In [ ]:
hh_reader = pd.read_stata(HH_FILE, iterator=True)
hh_cols = hh_reader.variable_labels().keys()

required = ["hv001", "hv270", "hv005"]

for col in required:
    if col not in hh_cols:
        raise ValueError(f"{col} not found in the Household Recode file.")

print("All required columns:", required)

## Read the Household recode and aggregate it to cluster (default target is hv270, weighted by hv005)

In [ ]:
hh = pd.read_stata(
    HH_FILE,
    columns=["hv001", "hv270", "hv005"],
    convert_categoricals=False
)

# hv001 is the DHS cluster ID
hh["cluster_id"] = hh["hv001"]

# Weighted wealth index (hv270), using DHS sample weights (hv005)
hh["w"] = hh["hv005"] / 1e6
hh["hv270_w"] = hh["hv270"] * hh["w"]

# Aggregate households to cluster-level target
cluster_target = (
    hh.groupby("cluster_id")
      .apply(lambda df: df["hv270_w"].sum() / df["w"].sum())
      .reset_index(name="wealth_index")
)

cluster_target.head()

## Merge GPS and target

In [ ]:
merged = cluster_df.merge(target_cluster, on="cluster_id", how="left")

print("Target coverage:", merged["target"].notna().mean())
merged.head()

## Nightlight data extraction

In [ ]:
print(src.crs)

In [ ]:
import math
from rasterio.windows import from_bounds
from tqdm import tqdm

def nl_mean_buffer(src, lon, lat, buffer_km=5):
    """Compute mean nightlights within a square buffer around (lon, lat), assuming EPSG:4326."""
    if src.crs is None:
        return np.nan

    lat_deg = buffer_km / 111.0
    lon_deg = buffer_km / (111.0 * max(math.cos(math.radians(lat)), 1e-6))

    left, right = lon - lon_deg, lon + lon_deg
    bottom, top = lat - lat_deg, lat + lat_deg

    win = from_bounds(left, bottom, right, top, transform=src.transform)
    data = src.read(1, window=win, masked=True, boundless=True)

    return float(data.mean()) if data.count() > 0 else np.nan

In [ ]:
RUN_FULL = True
BATCH_SIZE = 500

if RUN_FULL:
    batch_dir = os.path.join(RES_DIR, "batches")
    os.makedirs(batch_dir, exist_ok=True)

    full = merged.copy()
    n = len(full)
    n_batches = math.ceil(n / BATCH_SIZE)

    out_parts = []

    with rasterio.open(tif_path) as src:
        for bi in range(n_batches):

            start = bi * BATCH_SIZE
            end = min((bi + 1) * BATCH_SIZE, n)

            part = full.iloc[start:end].copy()

            part["nightlights_mean"] = [
                nl_mean_buffer(src, lon, lat, buffer_km=5)
                for lon, lat in zip(part["cluster_lon"], part["cluster_lat"])
            ]

            part_path = os.path.join(batch_dir, f"part_{bi:04d}.csv")
            part.to_csv(part_path, index=False)

            out_parts.append(part_path)
            print(f"Wrote batch {bi+1}/{n_batches}")

    all_df = pd.concat([pd.read_csv(p) for p in out_parts], ignore_index=True)

    # Save final cluster-level file
    final_path = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"
    all_df.to_csv(final_path, index=False)

    print("Saved final file:", final_path)
    print("Non-missing nightlights:", all_df["nightlights_mean"].notna().sum())

    all_df.head()

In [ ]:
# Final check
import pandas as pd

final_path = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"
check = pd.read_csv(final_path)

print("rows:", len(check))
print("unique clusters:", check["cluster_id"].nunique())
print("columns:", list(check.columns))